# vLLM vs Ollama smoke — Kaggle dual-T4 side-by-side

Runs a short PDDL-Copilot smoke matrix on **Qwen3.5:0.8B** under both Ollama and vLLM on a Kaggle dual-T4 session, with timers and per-cell metrics. Pins each backend to its own T4 (Ollama → GPU 0, vLLM → GPU 1) so neither evicts the other. Falls back to single-GPU sequential when only one T4 is present.

**Scope.** 0.8B only — the 27B (Q4_K_M / AWQ-INT4 ~17 GB) needs tensor-parallel across both T4s on vLLM, which Ollama doesn't natively split. For the 27B side-by-side, use `cluster-experimenting/run_smoke_vllm_vs_ollama.sbatch` on the BGU cluster.

Companion to that cluster sbatch. Decision rule documented in `development/CHANGELOG.md` (2026-05-09): migrate only if vLLM wall ≤ 0.7× Ollama AND tool-call pass-rate parity holds.


In [ ]:
# Parameters — edit and re-run.
MODEL_OLLAMA   = "Qwen3.5:0.8B"
MODEL_HF       = "Qwen/Qwen3.5-0.8B"
RUN_OLLAMA     = True                          # disable for vLLM-only run
RUN_VLLM       = True                          # disable for Ollama-only run
TASKS          = ["solve", "validate_plan"]    # subset; full = 5 tasks
PARTIAL_K      = 1                             # 1 fixture per domain (fast slice)
NUM_VARIANTS   = 1
CONDITIONS     = "both"                        # both | tools | no-tools
THINK          = "default"                     # default | on | off
CONCURRENCY    = 2
NUM_CTX        = 8192                          # 8K — T4 + concurrency=2 has the headroom; 16K is too tight
OLLAMA_GPU     = "0"                           # CUDA_VISIBLE_DEVICES for Ollama phase
VLLM_GPU       = "1"                           # CUDA_VISIBLE_DEVICES for vLLM phase
VLLM_PORT      = 8000
RESULTS_BASE   = ""                            # auto: /kaggle/working/results/vllm_smoke
RUN_TAG        = ""                            # auto-stamped if empty
EXPT_BRANCH    = "feat/vllm-smoke-probe"       # branch with --inference-backend

In [ ]:
import os, sys, time, subprocess, shlex, json, urllib.request
from pathlib import Path

PLATFORM = "kaggle" if Path("/kaggle").exists() else ("colab" if "google.colab" in sys.modules else "local")
print(f"Platform: {PLATFORM}")

WORK_DIR = "/kaggle/working/repos" if PLATFORM == "kaggle" else "/content"
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)

if not RESULTS_BASE:
    RESULTS_BASE = "/kaggle/working/results/vllm_smoke" if PLATFORM == "kaggle" else str(Path(WORK_DIR) / "results")
if not RUN_TAG:
    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_BASE = Path(RESULTS_BASE) / RUN_TAG
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
print("OUTPUT_BASE:", OUTPUT_BASE)

EXPT_DIR        = Path(WORK_DIR) / "pddl-copilot-experiments"
MARKETPLACE_DIR = Path(WORK_DIR) / "pddl-copilot"

def sh(cmd, **kw):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=kw.pop("check", True), text=True, **kw).returncode

def _clone(url, dst, branch=None):
    if dst.exists():
        print(f"exists: {dst}")
        return
    branch_arg = f"--branch {shlex.quote(branch)} " if branch else ""
    sh(f"git clone --depth 1 {branch_arg}{url} {shlex.quote(str(dst))}")

_clone("https://github.com/SPL-BGU/pddl-copilot-experiments.git", EXPT_DIR, branch=EXPT_BRANCH)
_clone("https://github.com/SPL-BGU/pddl-copilot.git",            MARKETPLACE_DIR)

In [ ]:
# OS deps: Java for ENHSP, zstd for the Ollama installer, python3-venv for plugin venvs.
if PLATFORM in ("colab", "kaggle"):
    sh("apt-get -qq update")
    sh("apt-get -qq install -y openjdk-17-jre-headless python3-venv python3.12-venv zstd", check=False)
    sh("update-java-alternatives -s java-1.17.0-openjdk-amd64 2>/dev/null || true", check=False)
sh("java -version")

# Python deps (harness + openai client + vllm).
sh(f"pip install --quiet -r {shlex.quote(str(EXPT_DIR / 'requirements.txt'))}")
if RUN_VLLM:
    sh("pip install --quiet 'vllm>=0.6.0'")

In [ ]:
# Pre-create plugin venvs so the first MCP spawn is instant
# (mirrors cluster-experimenting/setup_env.sh).
import shutil
for plugin in ("pddl-solver", "pddl-validator"):
    plug_dir = MARKETPLACE_DIR / "plugins" / plugin
    venv = plug_dir / ".venv"
    pip  = venv / "bin" / "pip"
    if venv.exists() and not pip.exists():
        shutil.rmtree(venv)
    if pip.exists():
        print(f"{plugin}: venv ready")
        continue
    print(f"{plugin}: creating venv...")
    sh(f"python3 -m venv {shlex.quote(str(venv))}")
    sh(f"{shlex.quote(str(pip))} install --quiet --upgrade pip")
    sh(f"{shlex.quote(str(pip))} install --quiet -r {shlex.quote(str(plug_dir / 'requirements.txt'))}")

In [ ]:
# GPU detect — Kaggle "GPU T4 x2" gives two visible GPUs. Falls back gracefully.
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
        text=True,
    )
    print(out)
    GPU_COUNT = len([l for l in out.strip().splitlines() if l])
except Exception as e:
    GPU_COUNT = 0
    print("nvidia-smi failed:", e)

print(f"GPU count: {GPU_COUNT}")
if RUN_OLLAMA and RUN_VLLM and GPU_COUNT < 2:
    print("WARNING: dual-T4 not detected — running both phases on GPU 0 sequentially.")
    print("         Numbers are still fair (each phase owns the GPU exclusively).")
    OLLAMA_GPU = VLLM_GPU = "0"

# Helpers shared by both phases.
def _run_smoke(cmd, cwd, log_path, label):
    """Stream `cmd` to the cell + tee to log_path; return (rc, wall_seconds)."""
    print(f"$ {' '.join(shlex.quote(c) for c in cmd)}")
    t0 = time.perf_counter()
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(log_path, "w") as logf:
        for line in proc.stdout:
            sys.stdout.write(line); logf.write(line)
    rc = proc.wait()
    wall = time.perf_counter() - t0
    print(f"\n[{label}] wall: {wall:.1f}s  rc={rc}")
    return rc, wall

def _stop(proc, name, timeout=15):
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            proc.kill()
    print(f"{name} stopped")

In [ ]:
# ====================  Phase A: Ollama  ====================
PHASE_A_DIR = OUTPUT_BASE / "ollama"
PHASE_A_DIR.mkdir(parents=True, exist_ok=True)
phase_a_wall = None

if RUN_OLLAMA:
    if subprocess.run("which ollama", shell=True, capture_output=True).returncode != 0:
        sh("curl -fsSL https://ollama.com/install.sh | sh")

    serve_env = {
        **os.environ,
        "OLLAMA_NUM_PARALLEL":   str(CONCURRENCY),
        "OLLAMA_KEEP_ALIVE":     "30m",
        "OLLAMA_CONTEXT_LENGTH": str(NUM_CTX),
        "CUDA_VISIBLE_DEVICES":  OLLAMA_GPU,
    }
    serve_log = PHASE_A_DIR / "ollama_serve.log"
    ollama_proc = subprocess.Popen(["ollama", "serve"],
                                   stdout=open(serve_log, "w"),
                                   stderr=subprocess.STDOUT, env=serve_env)

    def _ollama_up():
        try: urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); return True
        except Exception: return False

    for _ in range(60):
        if _ollama_up(): break
        time.sleep(1)
    if not _ollama_up():
        _stop(ollama_proc, "ollama")
        raise RuntimeError(f"ollama serve did not become ready; see {serve_log}")
    print(f"ollama up on GPU {OLLAMA_GPU}")
    sh(f"ollama pull {shlex.quote(MODEL_OLLAMA)}")

    cmd = [
        "python3", "run_experiment.py",
        "--marketplace-path", str(MARKETPLACE_DIR),
        "--models",           MODEL_OLLAMA,
        "--inference-backend", "ollama",
        "--tasks",            *TASKS,
        "--conditions",       CONDITIONS,
        "--num-variants",     str(NUM_VARIANTS),
        "--concurrency",      str(CONCURRENCY),
        "--num-ctx",          str(NUM_CTX),
        "--num-ctx-thinking", str(NUM_CTX),
        "--partial",          str(PARTIAL_K),
        "--output-dir",       str(PHASE_A_DIR),
    ]
    if THINK != "default":
        cmd += ["--think", THINK]

    rc, phase_a_wall = _run_smoke(cmd, str(EXPT_DIR), PHASE_A_DIR / "run.log", "Ollama")
    _stop(ollama_proc, "ollama")
else:
    print("RUN_OLLAMA=False — skipping Phase A")

In [ ]:
# ====================  Phase B: vLLM  ====================
PHASE_B_DIR = OUTPUT_BASE / "vllm"
PHASE_B_DIR.mkdir(parents=True, exist_ok=True)
phase_b_wall = None

if RUN_VLLM:
    serve_env = {**os.environ, "CUDA_VISIBLE_DEVICES": VLLM_GPU}
    serve_log = PHASE_B_DIR / "vllm_serve.log"
    serve_cmd = [
        "python3", "-m", "vllm.entrypoints.openai.api_server",
        "--model",                   MODEL_HF,
        "--port",                    str(VLLM_PORT),
        "--host",                    "0.0.0.0",
        "--max-model-len",           str(NUM_CTX),
        "--enable-auto-tool-choice",
        "--tool-call-parser",        "hermes",
        "--reasoning-parser",        "qwen3",
        "--gpu-memory-utilization",  "0.85",
        "--dtype",                   "float16",   # T4 lacks BF16
        "--enforce-eager",                          # skip CUDA graph compile (faster startup)
    ]
    print("$ " + " ".join(shlex.quote(c) for c in serve_cmd))
    vllm_proc = subprocess.Popen(serve_cmd, stdout=open(serve_log, "w"),
                                 stderr=subprocess.STDOUT, env=serve_env)

    def _vllm_up():
        try: urllib.request.urlopen(f"http://localhost:{VLLM_PORT}/v1/models", timeout=2); return True
        except Exception: return False

    print("Waiting for vLLM (up to 5 min for first run)...")
    ready = False
    for _ in range(150):
        if _vllm_up(): ready = True; break
        if vllm_proc.poll() is not None:
            print(f"vLLM died during startup; tail of {serve_log}:")
            sh(f"tail -50 {shlex.quote(str(serve_log))}", check=False)
            break
        time.sleep(2)
    if not ready:
        _stop(vllm_proc, "vllm")
        raise RuntimeError(f"vLLM did not become ready; see {serve_log}")
    print(f"vllm up on GPU {VLLM_GPU} (port {VLLM_PORT})")

    cmd = [
        "python3", "run_experiment.py",
        "--marketplace-path", str(MARKETPLACE_DIR),
        "--models",           MODEL_HF,
        "--inference-backend", "vllm",
        "--ollama-host",      f"http://localhost:{VLLM_PORT}",
        "--tasks",            *TASKS,
        "--conditions",       CONDITIONS,
        "--num-variants",     str(NUM_VARIANTS),
        "--concurrency",      str(CONCURRENCY),
        "--num-ctx",          str(NUM_CTX),
        "--num-ctx-thinking", str(NUM_CTX),
        "--partial",          str(PARTIAL_K),
        "--output-dir",       str(PHASE_B_DIR),
    ]
    if THINK != "default":
        cmd += ["--think", THINK]

    rc, phase_b_wall = _run_smoke(cmd, str(EXPT_DIR), PHASE_B_DIR / "run.log", "vLLM")
    _stop(vllm_proc, "vllm")
else:
    print("RUN_VLLM=False — skipping Phase B")

In [ ]:
# ====================  Comparison  ====================
def _read_trials(d):
    p = Path(d) / "trials.jsonl"
    if not p.exists(): return []
    return [json.loads(l) for l in p.open() if l.strip()]

def _stats(trials):
    if not trials: return None
    n = len(trials)
    succ = sum(1 for t in trials if t.get("success"))
    pt = sum((t.get("tokens") or {}).get("prompt", 0) for t in trials)
    ct = sum((t.get("tokens") or {}).get("completion", 0) for t in trials)
    et = sum((t.get("tokens") or {}).get("eval_duration_ns", 0) for t in trials) / 1e9
    return {"n": n, "succ": succ, "rate": succ / n if n else 0.0,
            "prompt_tok": pt, "completion_tok": ct, "decode_s": et,
            "decode_tok_s": (ct / et) if et > 0 else 0.0}

ollama_tr = _read_trials(PHASE_A_DIR) if RUN_OLLAMA else []
vllm_tr   = _read_trials(PHASE_B_DIR) if RUN_VLLM   else []
o, v = _stats(ollama_tr), _stats(vllm_tr)

print()
print("=" * 78)
print(f"  vLLM vs Ollama smoke summary  ({MODEL_OLLAMA} / {MODEL_HF})")
print("=" * 78)
print(f"  Tasks:        {TASKS}")
print(f"  Conditions:   {CONDITIONS}")
print(f"  Partial K:    {PARTIAL_K}    Concurrency: {CONCURRENCY}    num_ctx: {NUM_CTX}")
print()
print(f"  {'metric':24s} {'ollama':>14s} {'vllm':>14s} {'ratio (vllm/ollama)':>22s}")

def _row(name, oa, va, fmt="{:.1f}"):
    if oa is None or va is None or oa == 0:
        ratio = "n/a"
    else:
        ratio = f"{va/oa:.2f}x"
    oa_s = "n/a" if oa is None else fmt.format(oa)
    va_s = "n/a" if va is None else fmt.format(va)
    print(f"  {name:24s} {oa_s:>14s} {va_s:>14s} {ratio:>22s}")

_row("phase wall (s)",   phase_a_wall,            phase_b_wall)
_row("trials run",       o["n"]              if o else None, v["n"]              if v else None, fmt="{:d}")
_row("trials succeeded", o["succ"]           if o else None, v["succ"]           if v else None, fmt="{:d}")
_row("success rate",     o["rate"]           if o else None, v["rate"]           if v else None, fmt="{:.3f}")
_row("decode tok/s",     o["decode_tok_s"]   if o else None, v["decode_tok_s"]   if v else None, fmt="{:.1f}")
_row("prompt tokens",    o["prompt_tok"]     if o else None, v["prompt_tok"]     if v else None, fmt="{:d}")
_row("completion tok",   o["completion_tok"] if o else None, v["completion_tok"] if v else None, fmt="{:d}")
_row("decode time (s)",  o["decode_s"]       if o else None, v["decode_s"]       if v else None)

# Per-cell median trial wall (task × condition) — surfaces which cells
# benefit most from vLLM continuous-batching vs which ones don't move.
def _per_cell(trials):
    out = {}
    for t in trials:
        key = (t.get("task"), "tools" if t.get("with_tools") else "no-tools")
        wall = (t.get("tokens") or {}).get("total_duration_ns", 0) / 1e9
        out.setdefault(key, []).append(wall)
    return out

if ollama_tr and vllm_tr:
    pcA, pcB = _per_cell(ollama_tr), _per_cell(vllm_tr)
    print()
    print("  Per-cell median trial wall (s):")
    print(f"  {'task':18s} {'cond':10s} {'ollama':>10s} {'vllm':>10s} {'speedup':>10s}")
    for key in sorted(pcA.keys() & pcB.keys()):
        a = sorted(pcA[key])[len(pcA[key]) // 2]
        b = sorted(pcB[key])[len(pcB[key]) // 2]
        sp = f"{a/b:.2f}x" if b else "n/a"
        print(f"  {str(key[0]):18s} {str(key[1]):10s} {a:>10.2f} {b:>10.2f} {sp:>10s}")

print()
print("Outputs:")
if RUN_OLLAMA: print(f"  Ollama: {PHASE_A_DIR}")
if RUN_VLLM:   print(f"  vLLM:   {PHASE_B_DIR}")

## Notes

- **Why 0.8B only.** The 27B (Q4_K_M ~17 GB on Ollama / AWQ-INT4 ~17 GB on vLLM) needs tensor-parallel across both T4s for vLLM, and Ollama's multi-GPU split path is less battle-tested. For the 27B comparison, run `cluster-experimenting/run_smoke_vllm_vs_ollama.sbatch` on the BGU cluster (`rtx_6000:1` or `rtx_pro_6000:1`).

- **GPU pinning.** Ollama and vLLM are pinned to different T4s via `CUDA_VISIBLE_DEVICES` so neither evicts the other. Single-GPU sessions auto-fall back to GPU 0 sequentially — still fair, each phase has the GPU exclusively during its run.

- **`--enforce-eager` on vLLM.** Skips CUDA graph compilation (~30–60 s startup saving) at ~10–15% throughput cost. Fine for a smoke; remove for production-style runs where the saved ms/trial pays back.

- **`--dtype float16`.** T4 lacks native BF16; vLLM auto-converts the BF16 Qwen weights to FP16 on load. Numerically close enough for an indicative smoke, but the cluster smoke (rtx_6000 / Ada) keeps native BF16 — the cluster numbers are the authoritative gate.

- **Decision rule.** Per `development/CHANGELOG.md` 2026-05-09: migrate only if vLLM wall ≤ 0.7× Ollama AND tool-call pass rate within parity. The 0.8B numbers from this notebook are indicative of decode-throughput differences but won't capture the full picture — the 27B cluster smoke is what gates the migration.

- **Resume / re-run.** `run_experiment.py` reads `trials.jsonl` from each phase's output dir and skips completed trials, so re-executing a cell after a partial run only finishes what's missing. To start fresh, append `--no-resume` in the cmd list.
